<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python for Finance, 3rd Edition
## Appendix A · Linear Algebra and Optimization Toolkit

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Notebook Goals
This notebook mirrors the appendix examples in a Colab-ready format so that you can run, tweak, and extend them interactively.

### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted refactorings.
- Refer back to the book text for detailed explanations and context.

In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

## Vectors, Matrices, and Shapes
A vector is an ordered collection of numbers, and a matrix is a rectangular
collection of numbers.

In [ ]:
import numpy as np

In [ ]:
x = np.array([0.02, -0.01, 0.015])

In [ ]:
w = np.array([0.50, 0.30, 0.20])

In [ ]:
round(float(w @ x), 4), round(float(np.linalg.norm(x)), 4)

## Norms and Quadratic Forms
A norm measures the size of a vector.

In [ ]:
cov = np.array([
    [0.0400, 0.0180, 0.0120],
    [0.0180, 0.0625, 0.0200],
    [0.0120, 0.0200, 0.0900],
])

In [ ]:
var = float(w @ cov @ w)

In [ ]:
round(var, 5), round(float(np.sqrt(var)), 4)

## Linear Systems and Conditioning
A linear system has the form latexmath:[Ax = b].

In [ ]:
A = np.array([[1.0, 1.0], [1.02, 0.98]])

In [ ]:
b = np.array([1.0, 1.01])

In [ ]:
np.linalg.solve(A, b).round(4)

In [ ]:
round(float(np.linalg.cond(A)), 2)

## Eigenvalues and Factor Structure
Eigenvalues and eigenvectors describe directions in which a matrix acts like a
simple scalar multiplier.

In [ ]:
eigvals, eigvecs = np.linalg.eigh(cov)

In [ ]:
order = np.argsort(eigvals)[::-1]

In [ ]:
eigvals[order].round(5)

In [ ]:
eigvecs[:, order[0]].round(4)

## Least Squares as Projection
Least squares estimates coefficients by minimizing squared residuals.

In [ ]:
factors = np.array([
    [-0.02, 1.0],
    [0.01, 1.0],
    [0.03, 1.0],
    [0.04, 1.0],
])

In [ ]:
asset = np.array([-0.015, 0.012, 0.027, 0.041])

In [ ]:
beta, *_ = np.linalg.lstsq(factors, asset, rcond=None)

In [ ]:
beta.round(4)

## Constrained Optimization with SciPy
Many finance problems are optimization problems with constraints.

In [ ]:
from scipy import optimize

In [ ]:
mu = np.array([0.06, 0.08, 0.11])

In [ ]:
target = 0.085

In [ ]:
def objective(weights):
    return float(weights @ cov @ weights)

In [ ]:
cons = [
    {"type": "eq", "fun": lambda weights: np.sum(weights) - 1.0},
    {"type": "eq", "fun": lambda weights: float(mu @ weights) - target},
]

In [ ]:
res = optimize.minimize(
    objective,
    x0=np.repeat(1 / 3, 3),
    bounds=[(0, 1)] * 3,
    constraints=cons,
)

In [ ]:
res.x.round(4)

In [ ]:
round(float(mu @ res.x), 4), round(float(np.sqrt(res.fun)), 4)

## Symbolic Derivatives with SymPy
Analytical derivatives clarify how objectives change when inputs change.

In [ ]:
import sympy as sy

In [ ]:
w1, w2, c11, c12, c22 = sy.symbols("w1 w2 c11 c12 c22")

In [ ]:
q = c11 * w1**2 + 2 * c12 * w1 * w2 + c22 * w2**2

In [ ]:
[sy.diff(q, var) for var in (w1, w2)]

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">